# Patent Phrase Matching - DeBERTa Training
**Google Colab 실행용**

⚠️ Runtime → Change runtime type → **A100 GPU** 선택 후 실행

### 현재 설정 (검증된 0.858 구조 = 6801d20)
- CLS pooling, 단일 lr 2e-5, max_length=96, batch=8, epochs=4
- AMP(autocast+GradScaler), float32 모델
- **transformers==4.30.2 고정** (최신 버전은 deberta-v3 학습 깨짐)

### 실행 순서
1. 셀 1: Drive 마운트
2. 셀 2: Drive 용량 확인
3. 셀 3: 데이터 확인
4. 셀 4: 스크립트 다운로드 & 경로 패치
5. **셀 4.5: transformers 버전 고정 (필수!)**
6. 셀 5: 학습 시작
7. 셀 6 (선택): Drive 체크포인트 정리

In [ ]:
# 셀 1: Google Drive 마운트
from google.colab import drive
import os

drive.mount('/content/drive')

os.makedirs('/content/drive/MyDrive/aicodinggym_2/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/aicodinggym_2/hf_cache',    exist_ok=True)
print('✅ 디렉토리 준비 완료')

In [ ]:
# 셀 2: Drive 용량 확인
import subprocess

# 전체 Drive 용량
result = subprocess.run(['df', '-h', '/content/drive/MyDrive'], capture_output=True, text=True)
print('=== Drive 전체 용량 ===', result.stdout)

# aicodinggym_2 폴더 내 파일별 크기
ckpt_dir = '/content/drive/MyDrive/aicodinggym_2/checkpoints'
if os.path.exists(ckpt_dir):
    print('=== 체크포인트 파일 목록 ===')
    files = [(f, os.path.getsize(os.path.join(ckpt_dir, f))) for f in os.listdir(ckpt_dir)]
    files.sort(key=lambda x: x[1], reverse=True)
    total = 0
    for fname, size in files:
        print(f'  {fname}: {size/1e9:.2f} GB')
        total += size
    print(f'  합계: {total/1e9:.2f} GB')
else:
    print('체크포인트 폴더 없음')

# hf_cache 크기
hf_dir = '/content/drive/MyDrive/aicodinggym_2/hf_cache'
if os.path.exists(hf_dir):
    result2 = subprocess.run(['du', '-sh', hf_dir], capture_output=True, text=True)
    print(f'=== HF 캐시: {result2.stdout.split()[0]} ===')

In [ ]:
# 셀 3: 데이터 확인
data_path = '/content/drive/MyDrive/aicodinggym_2/us-patent-phrase-to-phrase-matching.zip'
if os.path.exists(data_path):
    size = os.path.getsize(data_path)
    print(f'✅ 데이터 확인: {size/1e6:.1f} MB')
else:
    print('❌ 데이터 없음!')
    print(f'   경로: {data_path}')
    print('   → Drive에 zip 파일을 업로드하세요')

In [ ]:
# 셀 4: 스크립트 다운로드 & 경로 패치
import re

!wget -q -O /content/deberta_finetune.py \
    'https://raw.githubusercontent.com/castlhoo/DSC204_us-patent-phrase-to-phrase-matching/main/deberta_finetune.py'

with open('/content/deberta_finetune.py', 'r') as f:
    code = f.read()

# 데이터 경로 패치
code = re.sub(
    r'"data_path"\s*:\s*"data/us-patent-phrase-to-phrase-matching\.zip"',
    '"data_path"          : "/content/drive/MyDrive/aicodinggym_2/us-patent-phrase-to-phrase-matching.zip"',
    code
)
# 체크포인트 경로 패치
code = re.sub(
    r'"ckpt_dir"\s*:\s*"checkpoints"',
    '"ckpt_dir"           : "/content/drive/MyDrive/aicodinggym_2/checkpoints"',
    code
)
# HF 캐시를 Drive에 저장 (재시작해도 다시 다운 안 받음)
code = code.replace(
    'os.environ["HF_HOME"] = os.path.expanduser("~/.hf_cache")',
    'os.environ["HF_HOME"] = "/content/drive/MyDrive/aicodinggym_2/hf_cache"'
)

with open('/content/deberta_finetune.py', 'w') as f:
    f.write(code)

print('✅ 경로 패치 완료')
!grep -n 'data_path\|ckpt_dir\|HF_HOME\|max_length\|batch_size\|epochs\|awp' /content/deberta_finetune.py | head -15

In [ ]:
# 셀 4.5: transformers 버전 고정 (최신 버전이 deberta-v3 학습을 깨뜨림)
# 최신 transformers(4.45+)의 새 로딩이 deberta-v3 disentangled attention을
# 제대로 로드 못 해서 학습이 안 됨(val_pearson~0.06). 안정 버전으로 고정.
# 셀 5는 별도 python 프로세스라 런타임 재시작 불필요!
!pip install -q transformers==4.30.2 tokenizers==0.13.3
import subprocess
r = subprocess.run(['python', '-c', 'import transformers; print(transformers.__version__)'],
                   capture_output=True, text=True)
print(f'✅ transformers 버전: {r.stdout.strip()}')

In [ ]:
# 셀 5: 학습 시작
!python /content/deberta_finetune.py

In [ ]:
# 셀 6 (선택): 이전 실패 체크포인트 정리
# 학습이 비정상 종료돼서 Drive에 쓸모없는 체크포인트가 남아있을 때 실행
import os, glob

ckpt_dir = '/content/drive/MyDrive/aicodinggym_2/checkpoints'
# progress.json에 완료 기록이 없는 fold의 임시 체크포인트만 삭제
import json
progress_path = os.path.join(ckpt_dir, 'progress.json')
completed = set()
if os.path.exists(progress_path):
    with open(progress_path) as f:
        completed = set(json.load(f).get('completed_folds', {}).keys())
print(f'완료된 fold: {completed}')

removed = []
for fpath in glob.glob(os.path.join(ckpt_dir, 'fold*_step.pt')) + \
             glob.glob(os.path.join(ckpt_dir, 'fold*_latest.pt')) + \
             glob.glob(os.path.join(ckpt_dir, 'fold*_best.pt')):
    fname = os.path.basename(fpath)
    fold_num = fname.split('fold')[1].split('_')[0]
    if fold_num not in completed:  # 미완료 fold의 임시 파일만 삭제
        size = os.path.getsize(fpath)
        os.remove(fpath)
        removed.append(f'{fname} ({size/1e9:.2f} GB 해제)')

if removed:
    print('삭제된 파일:')
    for r in removed:
        print(f'  {r}')
else:
    print('삭제할 파일 없음 (이미 깨끗함)')